# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading, overview, and analysis of the Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and schema records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# View and print summary metadata
meta = dataset.metadata
print(f"{meta.name}\n\n{meta.description}\n\nTags: {', '.join(meta.keywords) if meta.keywords else ''}")
if meta.personalSensitiveInformation:
    print(f"Personal sensitive fields: {', '.join(meta.personalSensitiveInformation)}")

## 2. Data Overview
List the available Record Sets and their Fields using their `@id` and names, as per the Croissant schema.

In [ ]:
# List Record Sets, fields, and columns by @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet: @id={rs['@id']}, name={rs['name'] if 'name' in rs else ''}")
    # List fields
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
        print(f"    - Field: @id={fid}")
    # List columns
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    for c in columns:
        cid = c['@id'] if isinstance(c, dict) and '@id' in c else c
        print(f"    - Column: @id={cid}")

## 3. Data Extraction
Load records from each Record Set by their `@id` into a DataFrame. All subsequent analyses reference Record Sets, Fields, and Columns by their `@id`.

In [ ]:
# Extract all record sets as DataFrames
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for RecordSet @id={rs_id}")
        else:
            print(f"No records found for RecordSet @id={rs_id}")
    except Exception as e:
        print(f"Error loading RecordSet @id={rs_id}: {e}")

# Show columns for each dataframe
for rs_id, df in dataframes.items():
    print(f"\nColumns for RecordSet @id={rs_id}:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data for a numeric field. All fields referenced by their `@id`. Adjust the `numeric_field_id` and RecordSet IDs to match the loaded data.

In [ ]:
# Choose a RecordSet and a numeric Field (ensure this matches the field @id from your data)
example_record_set_id = list(dataframes.keys())[0]  # Select the first non-empty RecordSet
df = dataframes[example_record_set_id]

# Find a suitable numeric field
numeric_field_id = None
for c in df.columns:
    # Example: match field id for GAD-7 total score or a similar numeric score from the survey
    if 'gad7' in c.lower() or 'phq9' in c.lower() or 'psq' in c.lower() or 'score' in c.lower():
        numeric_field_id = c
        break
if numeric_field_id is None:
    # Default to the first numeric-looking column
    for c in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break
        except Exception:
            continue

if numeric_field_id is None:
    print("No numeric field found in RecordSet for EDA.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    # Ensure the numeric field is numeric type
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # e.g., 75th percentile
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field if possible
    group_field = None
    for c in df.columns:
        if c != numeric_field_id and pd.api.types.is_string_dtype(df[c]):
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"\nAverage {numeric_field_id} grouped by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and its relationship to the chosen grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(y=df[group_field], x=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
We successfully loaded the Kilifi County Mental Health Survey dataset via its Croissant schema, explored its record sets, referenced fields and columns by their `@id`, and conducted basic EDA and visualization. Referencing schema entities by `@id` enables reproducible, schema-driven data workflows.